# Cash till open/close - YOLO26m segmentation

Fine-tune `yolo26m-seg` on the Roboflow project `ai-0emk1/cash-till-detect`.

Run the cells in order. Everything you would want to change lives in the
**Config** cell - names, paths, hyperparameters - and nothing below it is
hardcoded.

**Runtime > Change runtime type > GPU** before starting.

## 1. Install

In [ ]:
%pip install -q ultralytics roboflow

import torch, ultralytics
print('ultralytics', ultralytics.__version__)
print('torch      ', torch.__version__)
print('cuda       ', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU only - training will be far too slow)')

## 2. Config

Every output name is a variable here, so a second run with a different
`RUN_NAME` cannot overwrite the first one's weights.

In [ ]:
import os

# --- Roboflow source -------------------------------------------------
RF_API_KEY   = 'Y2xNSieFjnUfmZLpRypU'
RF_WORKSPACE = 'ai-0emk1'
RF_PROJECT   = 'cash-till-detect'
RF_VERSION   = 1
RF_FORMAT    = 'yolo26'

# --- where things go -------------------------------------------------
# Two roots, on purpose.
#
# LOCAL_ROOT holds everything training touches per batch. Drive is a FUSE
# mount: reading a dataset through it, every image, every epoch, is far
# slower than the local SSD. The raw export has to live here too, not just
# the split - the split is made of symlinks pointing back into the export,
# so an export on Drive would drag every read back onto Drive anyway.
#
# DRIVE_ROOT holds what has to outlive the session. /content is wiped when
# the runtime stops.
LOCAL_ROOT  = '/content'
DRIVE_ROOT  = '/content/drive/MyDrive/Liquir_Store/dev'

EXPORT_DIR  = f'{LOCAL_ROOT}/Cash-till-detect-{RF_VERSION}'  # raw roboflow export
DATASET_DIR = f'{LOCAL_ROOT}/till_seg'                       # the split we build
DATA_YAML   = f'{DATASET_DIR}/data.yaml'

# Checkpoints are written locally and synced to Drive at the end, rather
# than written straight to Drive. ultralytics rewrites last.pt AND best.pt
# after every epoch; at ~55 MB each that is ~11 GB pushed through the FUSE
# mount over 100 epochs. That is slow, and a stalled Drive write is a common
# way for a long Colab run to die at epoch 60 with nothing to show.
PROJECT_DIR = f'{LOCAL_ROOT}/runs/segment'
DRIVE_DIR   = f'{DRIVE_ROOT}/models'

# ...but 'synced at the end' is worthless if the session dies at epoch 60,
# so last.pt is pushed to Drive every SYNC_EVERY epochs as well. At ~55 MB a
# copy that is ~0.6 GB over a 100-epoch run instead of 11 GB, and it caps
# what a crash can destroy at SYNC_EVERY epochs (~19 min at this speed).
SYNC_EVERY  = 10

# --- run identity: every artefact below is named from this ------------
RUN_NAME    = 'till_seg_m960'
RUN_DIR     = f'{PROJECT_DIR}/{RUN_NAME}'
BEST_PT     = f'{RUN_DIR}/weights/best.pt'
LAST_PT     = f'{RUN_DIR}/weights/last.pt'
RESULTS_CSV = f'{RUN_DIR}/results.csv'
BEST_ONNX   = f'{RUN_DIR}/weights/best.onnx'
PRED_DIR    = f'{PROJECT_DIR}/{RUN_NAME}_pred'
ZIP_PATH    = f'{LOCAL_ROOT}/{RUN_NAME}.zip'
DRIVE_CKPT  = f'{DRIVE_DIR}/{RUN_NAME}_last.pt'   # the resume point
DRIVE_BEST  = f'{DRIVE_DIR}/{RUN_NAME}_best.pt'

# --- data split ------------------------------------------------------
# 'clip'   hold out a whole video          - honest, pessimistic
# 'source' random frames, augment-grouped  - optimistic, see cell 4
SPLIT_BY  = 'clip'
VAL_CLIP  = None      # None = auto-pick, see the split cell
VAL_FRAC  = 0.15      # SPLIT_BY='source' only
SEED      = 0

# --- model / training ------------------------------------------------
BASE_MODEL = 'yolo26m-seg.pt'
EPOCHS     = 100
IMGSZ      = 960      # not 640 - see the inspection cell for why
BATCH      = 8        # -1 lets ultralytics autosize to the GPU
WORKERS    = 2        # Colab disk is the bottleneck, not the CPU
PATIENCE   = 30
DEVICE     = 0 if __import__('torch').cuda.is_available() else 'cpu'

CLASS_NAMES = ['Close', 'Open']

# A polygon below either of these is a stray click, not an object.
MIN_POLY_PTS  = 3
MIN_POLY_AREA = 1e-5   # of the image; the smallest real instance is ~3.5e-3

from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

print('run     :', RUN_NAME)
print('device  :', DEVICE)
print('data    :', DATASET_DIR, '(local, fast)')
print('training:', RUN_DIR, '(local)')
print('kept at :', DRIVE_DIR, '(drive, survives the session)')

## 3. Download the dataset

In [ ]:
from roboflow import Roboflow

rf      = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
version = project.version(RF_VERSION)

print('project type:', project.type)
print('classes     :', project.classes)

dataset    = version.download(RF_FORMAT, location=EXPORT_DIR)
EXPORT_DIR = dataset.location
print('downloaded to', EXPORT_DIR)

for r, d, f in os.walk(EXPORT_DIR):
    if f:
        print(f'  {r.replace(EXPORT_DIR, ".")}: {len(f)} files')

## 4. Inspect before training

This export cannot be trained on as it ships. Three things, and the cell
below measures all of them rather than asserting them:

1. **No val split.** `data.yaml` points at `../valid/images` and
   `../test/images`; neither directory exists.
2. **The images are not independent.** Roboflow made 3 augmented copies
   (brightness / exposure / blur) of every source frame. Split at random and
   the same frame sits on both sides of the split wearing different lighting,
   so val measures memorisation.
3. **The frames come from very few video clips**, all one camera, one day.
   Even a leak-free frame split puts adjacent frames of one continuous shot
   on both sides, which is barely harder than sharing the frame itself.

It also measures how large the instances are, which is what picks `IMGSZ`.

In [ ]:
import collections, numpy as np

IMG_DIR = f'{EXPORT_DIR}/train/images'
LAB_DIR = f'{EXPORT_DIR}/train/labels'

def source_of(fname):
    """Frame identity - shared by the augmented copies of one frame."""
    return fname.split('_jpg.rf.')[0]

def clip_of(fname):
    """The video the frame was cut from."""
    return fname.split('_mp4-')[0]

files = sorted(f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.png')))
src   = collections.Counter(source_of(f) for f in files)
clips = collections.Counter(clip_of(f) for f in files)
print(f'{len(files)} images = {len(src)} source frames x '
      f'{collections.Counter(src.values()).most_common(1)[0][0]} augmented copies')
print(f'from {len(clips)} video clip(s):')
for k, v in clips.most_common():
    print(f'  {v:5d} imgs  {k}')

# polygons, class balance, instance size
n_pts, areas, cls = [], [], collections.Counter()
for f in os.listdir(LAB_DIR):
    for line in open(f'{LAB_DIR}/{f}'):
        p = line.split()
        if not p:
            continue
        cls[int(p[0])] += 1
        xy = np.array(p[1:], float).reshape(-1, 2)
        n_pts.append(len(xy))
        areas.append(np.ptp(xy[:, 0]) * np.ptp(xy[:, 1]))

print(f'\ninstances: ' + '  '.join(f'{CLASS_NAMES[k]}={v}' for k, v in sorted(cls.items())))
print(f'polygon points per instance: min={min(n_pts)} median={int(np.median(n_pts))} max={max(n_pts)}')
assert np.median(n_pts) >= 3, 'labels are boxes, not polygons - this is not a segmentation export'

# Degenerate annotations. In v1 there are 6, and they are not polygons at
# all: each is a 4-number detection line (cls cx cy w h) describing a box
# about 1x1 px - a stray click in the labelling tool. Each sits in a file
# that also holds a real 17-point mask, so the frame is kept and only the
# bad line is dropped when the split is written.
degen = [p for p, a in zip(n_pts, areas) if p < MIN_POLY_PTS or a < MIN_POLY_AREA]
print(f'\ndegenerate instances to drop: {len(degen)} of {len(n_pts)}'
      + (f'   (point counts: {sorted(set(degen))})' if degen else ''))

a = np.array(areas)
print(f'\ninstance bbox area, as a fraction of the image: '
      f'median={np.median(a):.4f}  smallest={a.min():.4f}')
for sz in (640, 960, 1280):
    print(f'  at imgsz={sz:4d}: median instance {np.sqrt(np.median(a))*sz:5.0f} px'
          f'  -> {np.sqrt(np.median(a))*sz/4:5.1f} px of mask (prototypes are stride 4)')

## 5. Build a split that does not leak

`SPLIT_BY='clip'` holds out an entire video. With only a handful of clips
that is a harsh test, and it is the only one here that answers *does this
work on footage it has not seen*. `SPLIT_BY='source'` groups the augmented
copies so no frame spans the split, but neighbouring frames of one shot
still do - read its number as an upper bound.

Images are symlinked, so the export stays the single copy on disk.

In [ ]:
import random, shutil

def label_path(stem):
    return f'{LAB_DIR}/{stem}.txt'

def clean_label(stem):
    """Label lines with the stray clicks removed. Returns (lines, n_dropped)."""
    keep, dropped = [], 0
    for line in open(label_path(stem)):
        p = line.split()
        if not p:
            continue
        xy = np.array(p[1:], float).reshape(-1, 2)
        if len(xy) < MIN_POLY_PTS or np.ptp(xy[:, 0]) * np.ptp(xy[:, 1]) < MIN_POLY_AREA:
            dropped += 1
            continue
        keep.append(line.rstrip('\n'))
    return keep, dropped

def class_counts(fs):
    n = collections.Counter()
    for f in fs:
        for line in clean_label(os.path.splitext(f)[0])[0]:
            n[int(line.split()[0])] += 1
    return n

key    = clip_of if SPLIT_BY == 'clip' else source_of
groups = collections.defaultdict(list)
for f in files:
    groups[key(f)].append(f)

if SPLIT_BY == 'clip':
    if VAL_CLIP is not None:
        val_keys = {VAL_CLIP}
    else:
        # Pick the held-out clip on evidence, not by hand: it has to contain
        # both classes (or the val number describes one class and says so
        # nowhere), and among those, take the one whose size is closest to
        # VAL_FRAC so val is neither a rounding error nor most of the data.
        ok = [k for k, fs in groups.items() if len(class_counts(fs)) == len(CLASS_NAMES)]
        if not ok:
            raise SystemExit('no single clip contains both classes - use SPLIT_BY="source"')
        val_keys = {min(ok, key=lambda k: abs(len(groups[k]) / len(files) - VAL_FRAC))}
    missing = val_keys - set(groups)
    if missing:
        raise SystemExit(f'clip not found: {missing}\navailable:\n  ' + '\n  '.join(sorted(groups)))
else:
    ks = sorted(groups)
    random.Random(SEED).shuffle(ks)
    val_keys = set(ks[:max(1, round(len(ks) * VAL_FRAC))])

split = {'train': [], 'val': []}
for k, fs in groups.items():
    split['val' if k in val_keys else 'train'].extend(fs)

if os.path.isdir(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
n_cleaned = n_dropped = n_emptied = 0
for s, fs in split.items():
    os.makedirs(f'{DATASET_DIR}/images/{s}')
    os.makedirs(f'{DATASET_DIR}/labels/{s}')
    for f in fs:
        stem = os.path.splitext(f)[0]
        os.symlink(f'{IMG_DIR}/{f}', f'{DATASET_DIR}/images/{s}/{f}')
        lines, dropped = clean_label(stem)
        dst = f'{DATASET_DIR}/labels/{s}/{stem}.txt'
        if dropped:
            # only this file needs rewriting; the rest stay symlinks so the
            # export remains the single copy of the data on disk
            n_cleaned += 1
            n_dropped += dropped
            n_emptied += not lines
            open(dst, 'w').write('\n'.join(lines) + ('\n' if lines else ''))
        else:
            os.symlink(label_path(stem), dst)

print(f'dropped {n_dropped} degenerate instances from {n_cleaned} files'
      + (f'; {n_emptied} files now have no labels at all (they become '
         'background images - check them)' if n_emptied else ''))

held_out = (str(sorted(val_keys)) if len(val_keys) <= 4
            else f'{len(val_keys)} groups, seed={SEED}')

with open(DATA_YAML, 'w') as fh:
    fh.write(f'# split_by={SPLIT_BY} val={held_out}\n')
    fh.write(f'path: {DATASET_DIR}\ntrain: images/train\nval: images/val\n\n')
    fh.write(f'nc: {len(CLASS_NAMES)}\nnames: {CLASS_NAMES}\n')

print(f'split by {SPLIT_BY}: {len(groups)} groups, held out {held_out}\n')
for s, fs in split.items():
    c   = class_counts(fs)
    tot = sum(c.values())
    print(f'  {s:5s} {len(fs):4d} imgs  '
          + '  '.join(f'{CLASS_NAMES[i]}={c[i]}' for i in range(len(CLASS_NAMES)))
          + f'   ({c[1]/tot*100:.0f}% {CLASS_NAMES[1]})')
    if min(c[i] for i in range(len(CLASS_NAMES))) == 0:
        print(f'  ! {s} is missing a class entirely - its metrics are meaningless')

print(open(DATA_YAML).read())

## 6. Train

Everything the run writes lands in `RUN_DIR`, on the local disk. Every
`SYNC_EVERY` epochs a copy of `last.pt` is pushed to Drive so a dead session
costs minutes rather than the whole run - see the resume cell below.

In [ ]:
import shutil
from ultralytics import YOLO

def sync_to_drive(trainer):
    """Push the checkpoints to Drive periodically. Registered as a callback
    so it runs inside the training loop, after the epoch's own save.

    Both files, not just last.pt. On resume ultralytics restores best_fitness
    from the checkpoint, so it only rewrites best.pt when an epoch beats that
    score - if the pre-crash best is never beaten, a best.pt that was left
    behind on the dead machine is never recreated."""
    epoch = trainer.epoch + 1                    # trainer.epoch is 0-based
    if epoch % SYNC_EVERY and epoch != trainer.epochs:
        return
    for src, dst in ((trainer.last, DRIVE_CKPT), (trainer.best, DRIVE_BEST)):
        if not os.path.exists(src):
            continue
        # copy to a temp name and rename, so a session that dies mid-copy
        # leaves the previous good checkpoint intact, not a half-written one
        shutil.copy(str(src), dst + '.tmp')
        os.replace(dst + '.tmp', dst)
    print(f'  [sync] epoch {epoch} -> {DRIVE_DIR}')

model = YOLO(BASE_MODEL)
model.add_callback('on_fit_epoch_end', sync_to_drive)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    patience=PATIENCE,
    seed=SEED,
    val=True,
    plots=True,
    # Roboflow already shipped 3 photometric copies of every frame, so some
    # colour jitter is baked into the data. Keep ours low rather than off:
    # the export varied brightness, exposure and blur - not hue.
    hsv_h=0.010, hsv_s=0.4, hsv_v=0.3,
    # One camera, one till, a handful of clips. Geometric augmentation is
    # doing most of the work of pretending this dataset is bigger than it is.
    degrees=5.0, translate=0.1, scale=0.4, fliplr=0.5,
    mosaic=1.0, close_mosaic=min(10, EPOCHS),
)

print('\nbest weights:', BEST_PT, os.path.exists(BEST_PT))
print('last weights:', LAST_PT, os.path.exists(LAST_PT))

## 6b. Resume after a dead session

Only if the run above was interrupted. Run cells 1, 2, 3, 4, 5 first to put
the data back on the fresh machine - `SEED` is fixed and the split is
deterministic, so it reproduces the same train/val split the checkpoint was
trained against. Then run this **instead of** cell 6.

The checkpoint carries its own hyperparameters, so nothing from the config
cell is re-applied here: to change `EPOCHS` or `BATCH` you have to start a
fresh run, not resume.

In [ ]:
import torch

assert os.path.exists(DRIVE_CKPT), f'no checkpoint on Drive at {DRIVE_CKPT}'

os.makedirs(f'{RUN_DIR}/weights', exist_ok=True)
shutil.copy(DRIVE_CKPT, LAST_PT)
if os.path.exists(DRIVE_BEST):
    shutil.copy(DRIVE_BEST, BEST_PT)   # so a best that is never beaten survives

ck = torch.load(LAST_PT, map_location='cpu', weights_only=False)
print(f'resuming from epoch {ck["epoch"] + 1}')

model = YOLO(LAST_PT)
model.add_callback('on_fit_epoch_end', sync_to_drive)
results = model.train(resume=True)

## 7. Validate

Mask metrics, per class. `Open` is the rarer class and the one the alert
would fire on, so read its recall before the headline mAP.

In [ ]:
best = YOLO(BEST_PT)
m    = best.val(data=DATA_YAML, imgsz=IMGSZ, device=DEVICE, split='val',
                project=PROJECT_DIR, name=f'{RUN_NAME}_val', exist_ok=True)

print(f'\nmask  mAP50-95 {m.seg.map:.3f}   mAP50 {m.seg.map50:.3f}')
print(f'box   mAP50-95 {m.box.map:.3f}   mAP50 {m.box.map50:.3f}\n')
for i, c in enumerate(m.seg.ap_class_index):
    print(f'  {best.names[c]:<6} mAP50={m.seg.ap50[i]:.3f}  '
          f'P={m.seg.p[i]:.3f}  R={m.seg.r[i]:.3f}')

import pandas as pd
df = pd.read_csv(RESULTS_CSV)
df.columns = [c.strip() for c in df.columns]
df.tail(5)

In [ ]:
# training curves
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg']:
    p = f'{RUN_DIR}/{f}'
    if os.path.exists(p):
        print(f)
        display(Image(filename=p, width=900))

## 8. Predict on held-out images

In [ ]:
from IPython.display import Image, display

val_imgs = sorted(os.listdir(f'{DATASET_DIR}/images/val'))
sample   = [f'{DATASET_DIR}/images/val/{f}' for f in val_imgs[:8]]

preds = best.predict(sample, imgsz=IMGSZ, device=DEVICE, conf=0.25,
                     save=True, project=PROJECT_DIR,
                     name=os.path.basename(PRED_DIR), exist_ok=True)

for r in preds:
    got = [(best.names[int(c)], float(s)) for c, s in zip(r.boxes.cls, r.boxes.conf)]
    print(os.path.basename(r.path), '->', got or 'nothing detected')

for f in sorted(os.listdir(PRED_DIR))[:4]:
    display(Image(filename=f'{PRED_DIR}/{f}', width=800))

## 9. Export and save to Drive

ONNX so the model drops into the same runtime as the rest of the system.

Everything so far was written to `/content`, which Colab wipes when the
session ends. This is the cell that makes the run survive, so run it before
closing the tab.

In [ ]:
onnx_path = best.export(format='onnx', imgsz=IMGSZ, opset=12,
                        simplify=True, dynamic=False)
print('exported:', onnx_path)
BEST_ONNX = str(onnx_path)

In [ ]:
# The weights on their own do not say what produced them, so the data.yaml
# that defined the split goes into the archive next to them.
shutil.copy(DATA_YAML, f'{RUN_DIR}/data.yaml')
shutil.make_archive(ZIP_PATH[:-4], 'zip', RUN_DIR)   # weights, plots, results.csv
print(f'{ZIP_PATH}  {os.path.getsize(ZIP_PATH)/1e6:.1f} MB')

# One copy of each artefact on Drive, prefixed with RUN_NAME so a second run
# under a different name lands beside this one instead of on top of it.
for p in [BEST_PT, LAST_PT, BEST_ONNX, RESULTS_CSV, ZIP_PATH]:
    if not os.path.exists(p):
        print('missing, skipped:', p)
        continue
    dst = f'{DRIVE_DIR}/{RUN_NAME}_{os.path.basename(p)}'
    shutil.copy(p, dst)
    print(f'saved {dst}  ({os.path.getsize(dst)/1e6:.1f} MB)')

print('\ncontents of', DRIVE_DIR)
for f in sorted(os.listdir(DRIVE_DIR)):
    print(' ', f)

In [ ]:
# optional: also pull the archive down to your own machine
from google.colab import files
files.download(ZIP_PATH)